# Quantum-Inspired Reservoir (QIR) Layer

This notebook implements the core **Quantum-Inspired Reservoir Layer** in PyTorch.
It uses **orthogonal unitary matrix dynamics** ($U^\dagger U = I$) via `scipy.stats.ortho_group` to simulate quantum unitary evolution without explicit quantum state gate simulation.

In [ ]:
import torch
import torch.nn as nn
import scipy.stats
import numpy as np

class QuantumInspiredReservoir(nn.Module):
    """
    Quantum-Inspired Reservoir Layer using orthogonal unitary matrix dynamics
    for efficient temporal sequence encoding.
    """
    def __init__(self, input_dim, reservoir_dim, spectral_radius=0.95, leak_rate=0.3):
        super(QuantumInspiredReservoir, self).__init__()
        self.input_dim = input_dim
        self.reservoir_dim = reservoir_dim
        self.leak_rate = leak_rate
        
        # Initialize input weights
        self.W_in = nn.Parameter(
            torch.FloatTensor(reservoir_dim, input_dim).uniform_(-0.1, 0.1),
            requires_grad=False
        )
        
        # Generate Orthogonal Unitary Matrix to mimic Quantum Unitary Evolution
        U = scipy.stats.ortho_group.rvs(dim=reservoir_dim)
        W_res = torch.tensor(U, dtype=torch.float32)
        
        # Scale by spectral radius to enforce echo-state property
        max_eig = torch.max(torch.abs(torch.linalg.eigvals(W_res)))
        self.W_res = nn.Parameter((W_res / max_eig) * spectral_radius, requires_grad=False)

    def forward(self, x, h_prev=None):
        """
        x: Input tensor of shape (batch_size, sequence_length, input_dim)
        Returns: All reservoir states across time
        """
        batch_size, seq_len, _ = x.shape
        if h_prev is None:
            h_prev = torch.zeros(batch_size, self.reservoir_dim, device=x.device)
            
        states = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            # Non-linear unitary dynamics state update
            pre_act = torch.matmul(x_t, self.W_in.T) + torch.matmul(h_prev, self.W_res.T)
            h_state = (1 - self.leak_rate) * h_prev + self.leak_rate * torch.tanh(pre_act)
            states.append(h_state.unsqueeze(1))
            h_prev = h_state
            
        return torch.cat(states, dim=1)

# Test instantiation of QIR layer
batch_size, seq_len, input_dim, reservoir_dim = 16, 50, 2, 64
qir = QuantumInspiredReservoir(input_dim=input_dim, reservoir_dim=reservoir_dim)
dummy_input = torch.randn(batch_size, seq_len, input_dim)
output_states = qir(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"QIR Layer Output shape: {output_states.shape}")